In [1]:
# 2025.11.21 경주님 함수 수정 적용 후

import os
import sys

# 현재 작업 디렉토리 기준으로 상위 1단계 폴더를 루트로 설정
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# print("프로젝트 루트로 설정된 경로:", project_root)

In [ ]:
# 데이터 확인하기 2025.11.21 zero_count_rate > 95% 이상인 컬럼 제거 후 
import pandas as pd
import numpy  as np

from sklearn.preprocessing import StandardScaler # 데이터 전처리용

import matplotlib.pyplot as plt
import seaborn as sns

import importlib
from utils import preprocessing, user_utils

# 모듈 reload
importlib.reload(preprocessing)
importlib.reload(user_utils)

from utils.preprocessing import load_data, split_features_target, scale_data, data_split, remove_zero_columns2
from utils.user_utils    import get_model_train_eval

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 데이터 로딩
train, test = load_data()
# 데이터 할당
X_features, y_labels = split_features_target(train) # ID와 TARGET 모두 제거하고 X_train 만들기
X_test     = test.drop(columns=['ID'], axis=1) # test 데이터에서도 ID 제거 

In [4]:
# Data 전처리 1. zero_count_rate이 95%인 컬럼 제거하기 
X_features, X_test = remove_zero_columns2(X_features, X_test, 0.95)


Train Data Analysis (Threshold: 95.0% )
Train rows: 76,020, columns: 369
Test rows: 75,818, columns: 369

                                   Train Summary (zero_count 내림차순)                                    
                   ColumnName  na_Sum  nUnique          mode  modeFreq modeFreqRate  zero_count zero_count_rate_display
saldo_medio_var13_medio_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
                     ind_var2       0        1      0.000000     76020      100.00%       76020                 100.00%
        num_reemb_var33_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
    num_trasp_var17_out_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
    num_trasp_var33_out_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
              saldo_var2_ult1       0        1      0.000000     76020

In [5]:
# Data 전처리 2. var3 의 최소값 -99999 를 최빈값으로 변경하기
X_features['var3'] = X_features['var3'].replace(-999999, 2)

In [6]:
# 스케일링
X_train_scaled, X_test_scaled, scaler = scale_data(X_features, X_test)

In [7]:
# 학습/테스트 데이터 분리
X_train, X_val, y_train, y_val = data_split(
  X_features, 
  y_labels,
)

In [ ]:
# 1. XGBoost, 95%

# 99% 와 동일한 하이퍼파라미터 적용
from xgboost import XGBClassifier

# 모델 이름 생성
model_name = 'XGBoost_95per_basic'

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=23
)

# 함수 이용 (경주님 함수 수정 후 적용)
get_model_train_eval(xgb, model_name, X_train, X_val, y_train, y_val)

# AUC: 0.8421, 정확도: 0.9603, 정밀도: 0.4286, 재현율: 0.0050, F1: 0.0099
# 오차행렬:
# [[14598     4]
#  [  599     3]]
# 실행 시간: 2.3912107944488525

✓ 모델 저장 완료: models\XGBoost_95per_basic.pkl
  파일 크기: 1.53 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8421, 정확도: 0.9603, 정밀도: 0.4286, 재현율: 0.0050, F1: 0.0099
오차행렬:
[[14598     4]
 [  599     3]]
실행 시간: 2.3912107944488525


In [ ]:
# 2. LogisticRegression 95%

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

# 모델 이름 생성
model_name = 'LogisticRegression_95per_basic'

log_reg = LogisticRegression(
    max_iter=1000,        # 반복 횟수 충분히 크게
    solver='lbfgs',       # 최적화 알고리즘
    random_state=42,
    n_jobs=-1
)

# 함수 이용
get_model_train_eval(log_reg, model_name, X_train, X_val, y_train, y_val)

# AUC: 0.5529, 정확도: 0.9604, 정밀도: 0.5000, 재현율: 0.0033, F1: 0.0066
# 오차행렬:
# [[14600     2]
#  [  600     2]]
# 실행 시간: 15.08945631980896

✓ 모델 저장 완료: models\LogisticRegression_95per_basic.pkl
  파일 크기: 0.00 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.5529, 정확도: 0.9604, 정밀도: 0.5000, 재현율: 0.0033, F1: 0.0066
오차행렬:
[[14600     2]
 [  600     2]]
실행 시간: 15.08945631980896


In [ ]:
# XGBoost 95%
# HyperOpt 최적 하이퍼파라미터 탐색

from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

# 목적 함수 정의
def objective(params):
    model = XGBClassifier(
        n_estimators=int(params['n_estimators']),
        max_depth=int(params['max_depth']),
        learning_rate=params['learning_rate'],
        subsample=params['subsample'],
        colsample_bytree=params['colsample_bytree'],
        random_state=23,
        n_jobs=-1,
        use_label_encoder=False,
        eval_metric='logloss'
    )
    
    # 학습
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_val)[:, 1]
    
    # 평가 지표 (ROC-AUC 기준)
    auc = roc_auc_score(y_val, y_proba)
    
    return {
        'loss': -auc,   # HyperOpt는 최소화 → 음수로 변환
        'status': STATUS_OK,
        'auc': auc
    }

# 탐색 공간 정의
space = {
    'n_estimators': hp.quniform('n_estimators', 100, 1000, 50),   # 트리 개수
    'max_depth': hp.quniform('max_depth', 3, 10, 1),              # 트리 깊이
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),      # 학습률
    'subsample': hp.uniform('subsample', 0.6, 1.0),               # 샘플링 비율
    'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 1.0)  # 피처 샘플링 비율
}

# Trials 객체 (결과 저장)
trials = Trials()

# 최적화 실행
best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=30,   # 시도 횟수 (필요시 늘리기)
    trials=trials,
    rstate=np.random.default_rng(42)
)

print("Best Hyperparameters:", best)
# 100%|██████████| 30/30 [01:12<00:00,  2.43s/trial, best loss: -0.8527506813111206
# Best Hyperparameters: {'colsample_bytree': np.float64(0.7641293893463502), 
# 'learning_rate': np.float64(0.017169611817710123), 
# 'max_depth': np.float64(6.0), 
# 'n_estimators': np.float64(350.0), 
# 'subsample': np.float64(0.8352914228773503)}

  0%|          | 0/30 [00:00<?, ?trial/s, best loss=?]

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\hyperopt\atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\xgboost\training.py:199: UserWarning: [17:48:39] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



  3%|▎         | 1/30 [00:04<02:14,  4.62s/trial, best loss: -0.7969043288567852]

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\xgboost\training.py:199: UserWarning: [17:48:43] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



  7%|▋         | 2/30 [00:09<02:10,  4.66s/trial, best loss: -0.8378970977898172]

...

 93%|█████████▎| 28/30 [01:09<00:03,  1.72s/trial, best loss: -0.8527506813111206]

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\xgboost\training.py:199: UserWarning: [17:49:48] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



 97%|█████████▋| 29/30 [01:11<00:01,  1.76s/trial, best loss: -0.8527506813111206]

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\xgboost\training.py:199: UserWarning: [17:49:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



100%|██████████| 30/30 [01:12<00:00,  2.43s/trial, best loss: -0.8527506813111206]
Best Hyperparameters: {'colsample_bytree': np.float64(0.7641293893463502), 'learning_rate': np.float64(0.017169611817710123), 'max_depth': np.float64(6.0), 'n_estimators': np.float64(350.0), 'subsample': np.float64(0.8352914228773503)}


In [ ]:
# XGBoost 95%
# HyperOpt 적용

'''
- HyperOpt 실행 → best 딕셔너리에 최적 하이퍼파라미터 저장
- 딕셔너리 변환 → best_params로 정리 (형변환 필요: int)
- XGBClassifier 생성 → 최적 파라미터 적용
- get_model_train_eval 실행 → 학습/검증 결과 확인
'''

from xgboost import XGBClassifier

# HyperOpt 결과(best) 예시 출력
print("Best Hyperparameters:", best)

# best 딕셔너리에서 값 꺼내기
best_params = {
    'n_estimators': int(best['n_estimators']),
    'max_depth': int(best['max_depth']),
    'learning_rate': best['learning_rate'],
    'subsample': best['subsample'],
    'colsample_bytree': best['colsample_bytree']
}

# 최적 하이퍼파라미터로 모델 생성
model_name = 'XGBoost_95per_hyperopt'

xgb_best = XGBClassifier(
    **best_params,
    random_state=23,
    n_jobs=-1,
    use_label_encoder=False,
    eval_metric='logloss'
)

# 학습/평가 실행
# get_model_train_eval 함수에 연결
get_model_train_eval(xgb_best, model_name, X_train, X_val, y_train, y_val)

# Best Hyperparameters: {
    # 'colsample_bytree': np.float64(0.7641293893463502), 
    # 'learning_rate': np.float64(0.017169611817710123), 
    # 'max_depth': np.float64(6.0), 
    # 'n_estimators': np.float64(350.0), 
    # 'subsample': np.float64(0.8352914228773503)
# }    
# AUC: 0.8528, 정확도: 0.9604, 정밀도: 0.5000, 재현율: 0.0050, F1: 0.0099
# 오차행렬:
# [[14599     3]
#  [  599     3]]
# 실행 시간: 1.79510498046875

Best Hyperparameters: {'colsample_bytree': np.float64(0.7641293893463502), 'learning_rate': np.float64(0.017169611817710123), 'max_depth': np.float64(6.0), 'n_estimators': np.float64(350.0), 'subsample': np.float64(0.8352914228773503)}


c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\xgboost\training.py:199: UserWarning: [17:56:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✓ 모델 저장 완료: models\XGBoost_95per_hyperopt.pkl
  파일 크기: 1.17 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8528, 정확도: 0.9604, 정밀도: 0.5000, 재현율: 0.0050, F1: 0.0099
오차행렬:
[[14599     3]
 [  599     3]]
실행 시간: 1.79510498046875


In [ ]:
# Logistic Regression 95%
# HyperOpt 최적 하이퍼파라미터 탐색

from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np

# 목적 함수 정의
def objective(params):
    # Logistic Regression 모델 생성
    model = LogisticRegression(
        max_iter=1000,
        solver='lbfgs',
        random_state=42,
        n_jobs=-1,
        C=params['C'],                # 정규화 강도
        penalty=params['penalty']     # 규제 방식
    )
    
    # 학습
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)[:, 1]
    
    # 평가 지표 (ROC-AUC 기준)
    auc = roc_auc_score(y_val, y_proba)
    
    return {
        'loss': -auc,   # HyperOpt는 최소화 → 음수로 변환
        'status': STATUS_OK,
        'auc': auc
    }

# 탐색 공간 정의
space = {
    'C': hp.loguniform('C', np.log(0.001), np.log(10)),   # 정규화 강도
    'penalty': hp.choice('penalty', ['l2'])               # LogisticRegression은 lbfgs에서 l2만 지원
}

# Trials 객체 (결과 저장)
trials = Trials()

# 최적화 실행
best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=30,   # 시도 횟수
    trials=trials,
    rstate=np.random.default_rng(42)
)

print("Best Hyperparameters:", best)

# 100%|██████████| 30/30 [06:47<00:00, 13.60s/trial, best loss: -0.602404622131133]
# Best Hyperparameters: {'C': np.float64(1.0445023972192211), 'penalty': np.int64(0)}

100%|██████████| 30/30 [06:47<00:00, 13.60s/trial, best loss: -0.602404622131133]
Best Hyperparameters: {'C': np.float64(1.0445023972192211), 'penalty': np.int64(0)}


In [ ]:
from sklearn.linear_model import LogisticRegression

# HyperOpt 결과(best) 출력
print("Best Hyperparameters:", best)

# best 딕셔너리에서 값 꺼내기
best_params = {
    'C': best['C'],                       # 정규화 강도
    'penalty': ['l2'][best['penalty']]     # hp.choice로 선택된 penalty 인덱스를 실제 값으로 변환
}

# 최적 하이퍼파라미터로 모델 생성
model_name = 'LogisticRegression_95per_hyperopt'

log_reg_best = LogisticRegression(
    **best_params,
    max_iter=1000,
    solver='lbfgs',
    random_state=42,
    n_jobs=-1
)

# 학습/평가 실행
get_model_train_eval(log_reg_best, model_name, X_train, X_val, y_train, y_val)

# Best Hyperparameters: {
    # 'C': np.float64(1.0445023972192211), 
    # 'penalty': np.int64(0)
# }
# AUC: 0.6024, 정확도: 0.9603, 정밀도: 0.0000, 재현율: 0.0000, F1: 0.0000
# 오차행렬:
# [[14601     1]
#  [  602     0]]
# 실행 시간: 13.2692711353302

Best Hyperparameters: {'C': np.float64(1.0445023972192211), 'penalty': np.int64(0)}
✓ 모델 저장 완료: models\LogisticRegression_95per_hyperopt.pkl
  파일 크기: 0.00 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.6024, 정확도: 0.9603, 정밀도: 0.0000, 재현율: 0.0000, F1: 0.0000
오차행렬:
[[14601     1]
 [  602     0]]
실행 시간: 13.2692711353302
